## Лабораторная работа 1. Введение в ML

В этой лабораторной вы:

- познакомитесь с базовыми библиотеками для работы с табличными данными — `numpy` и `pandas`
- поближе посмотрите на простейшие задачи машинного обучения: классификацию и регрессию
- попробуете несколько метрик и поймёте, почему выбор метрики это важно
- обучите несколько простых моделей
- увидите связь между сложностью модели и переобучением
- убедитесь, что без данных всё тлен

Загрузка самых базовых библиотек:

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split

### [NumPy](https://numpy.org/doc/stable/user/index.html)

С 1995 numeric, с 2006 NumPy — «Numerical Python extensions» или просто «NumPy»

Возможности библиотеки NumPy:
* работать с многомерными массивами (таблицами)
* быстро вычислять математические функций на многомерных массивах

Ядро пакета NumPy — объект [ndarray](https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.html)

**Важные отличия** между NumPy arrays и Python sequences: 
* NumPy array имеет фиксированную длину, которая определяется в момент его создания (в отличие от Python lists, которые могут расти динамически)
* Элементы в NumPy array должны быть одного типа
* Можно выполнять операции непосредственно над NumPy arrays

**Скорость** NumPy достигается с помощью:
* реализации на C
* векторизации и броадкастинга (broadcasting). Например, произведение массивов совместимых форм.

Теперь давайте разберёмся подробнее и сделаем что-нибудь приятное и полезное в `numpy`!

### Индексация

В NumPy работает привычная индексация Python, ура! Включая использование отрицательных индексов и срезов (slices)

<div class="alert alert-info">
<b>Замечание 1:</b> Индексы и срезы в многомерных массивах не нужно разделять квадратными скобками, 
т.е. вместо <b>matrix[i][j]</b> нужно использовать <b>matrix[i, j]</b>. Первое тоже работает, но сначала выдаёт строку i, потом элемент j в ней. 
</div>

<div class="alert alert-danger">
<b>Замечание 2:</b> Срезы в NumPy создают view, а не копии, как в случае срезов встроенных последовательностей Python (string, tuple and list).
</div>

In [38]:
ones_matrix = np.ones((5, 5))
ones_submatrix_view = ones_matrix[::2,::2] # creates a view, not copy
ones_matrix[::2,::2] = np.zeros((3, 3))
ones_submatrix_view

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

### Ссылка на Яндекс.Контест

Решения и ответы в задачах, расположенных ниже, загружайте в контест на автоматическую проверку:
https://contest.yandex.ru/contest/39295/enter

**1. (1 балл)** Реализуйте функцию, принимающую на вход два одномерных массива `first_array` и `second_array` и возвращающую матрицу, в которой первый массив соответствует первому столбцу матрицы, второй — второму.

Вероятно первое, что приходит вам на ум, это конкатенация и транспонирование:

In [39]:
def construct_matrix(first_array, second_array):
    """
    Construct matrix from pair of arrays
    :param first_array: first array
    :param second_array: second array
    :return: constructed matrix
    """
    
    return np.column_stack((first_array, second_array))

In [40]:
construct_matrix(np.array([1,2]),np.array([3,4]))

array([[1, 3],
       [2, 4]])

(в скобках заметим, что конкатенировать можно vertically, horizontally, depth wise методами vstack, hstack, dstack по трём осям (0, 1 и 2, соотвественно), либо в общем случае `np.concatenate` — поиграйтесь ниже с прекрасным примером четырёхмерной точки, чтобы точно всё для себя понять)

In [41]:
p = np.arange(1).reshape([1, 1, 1, 1])
p

array([[[[0]]]])

In [42]:
print("vstack: ", np.vstack((p, p)).shape)
print("hstack: ", np.hstack((p, p)).shape)
print("dstack: ", np.dstack((p, p)).shape)

vstack:  (2, 1, 1, 1)
hstack:  (1, 2, 1, 1)
dstack:  (1, 1, 2, 1)


In [43]:
np.concatenate((p, p), axis=3).shape

(1, 1, 1, 2)

Но, поскольку операция транспонирования [делает массив non-contiguous](https://numpy.org/doc/stable/user/basics.copies.html#other-operations), мы в этой задаче **запретим** ей пользоваться и порекомедуем воспользоваться, например, методом [reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html).

**2. (1 балл)** Реализуйте функцию, принимающую на вход массив целых неотрицательных чисел `nums` и возвращающую самый частый элемент массива.

In [44]:
def most_frequent(nums):
    """
    Find the most frequent value in an array
    :param nums: array of ints
    :return: the most frequent value
    """
    values, counts = np.unique(nums, return_counts=True)
    return values[np.argmax(counts)]


### Переходим к работе с данными

Прежде всего, загрузим данные и сделаем из них красивые pandas-таблички. Они взяты из параллели RecSys соревнования https://yandex.ru/cup/ml/. Но мы будем иметь дело не со всеми данными, а только с их частью. Данные у нас будут про заведения общественного питания (больше бюрократический терминологии!)

Задачей будет **предсказание среднего чека** (average_bill) по некоторым другим свойствам заведения.

In [45]:
base = './data/'

In [46]:
data = pd.read_csv(base + 'organisations.csv')
features = pd.read_csv(base + 'features.csv')
rubrics = pd.read_csv(base + 'rubrics.csv')

В основном мы будем работать с табличкой `data`; остальное вам может пригодиться, если вы захотите знать, какое содержание стоит за кодами признаков.

## Изучение данных

Посмотрите на данные. В этом вам поможет метод ``head`` pandas-таблички.

In [47]:
# <Your code here>
data.head()

features.head()

# rubrics.head()

,feature_id,feature_name
0,1,prepress_and_post_printing_processing
1,40,products
2,54,printing_method
3,77,fuel
4,79,shop


Полезно посмотреть внимательнее на то, с какими признаками нам предстоит работать.

* **org_id** вам не понадобится;
* **city** - город, в котором находится заведение (``msk`` или ``spb``);
* **average_bill** - средний чек в заведении - он будет нашим таргетом;
* **rating** - рейтинг заведения;
* **rubrics_id** - тип заведения (или несколько типов). Соответствие кодов каким-то человекочитаемым типам живёт в табличке ``rubrics``
* **features_id** - набор неких фичей заведения. Соответствие кодов каким-то человекочитаемым типам живёт в табличке ``features``

Обратите внимание, что **rubrics_id** и **features_id** - это не списки, а разделённые пробелами строки. Когда вам захочется работать с отдельными фичами из мешка фичей для данного заведения, вам придётся всё-таки превратить их в списки (здесь поможет метод `split` для строк).

Чтобы быстро восстанавливать по рубрикам и фичам их нормальные названия, сделайте словари вида ``код_фичи:название_фичи``

In [48]:
# <Your code here>
# Создадим словарь для рубрик
rubric_dict = dict(zip(rubrics['rubric_id'], rubrics['rubric_name']))

# Создадим словарь для фичей
feature_dict = dict(zip(features['feature_id'], features['feature_name']))


Посмотрим, какими бывают типы заведений:

In [49]:
rubric_dict

{30519: 'Булочная, пекарня',
 30770: 'Бар, паб',
 30771: 'Быстрое питание',
 30774: 'Кафе',
 30775: 'Пиццерия',
 30776: 'Ресторан',
 30777: 'Столовая',
 31286: 'Спортбар',
 31350: 'Кондитерская',
 31375: 'Суши-бар',
 31401: 'Кальян-бар',
 31495: 'Кофейня',
 3108292683: 'Бар безалкогольных напитков',
 3501514558: 'Фудкорт',
 3501750896: 'Кофе с собой'}

Мы что-то поняли про признаки, которыми нам предстоит пользоваться. Теперь время посмотреть на таргет. Вооружившись функциями ``hist`` и ``scatter`` из библиотеки ``matplotlib``, а также методом ``isna`` для pandas-таблиц разберитесь, какие значения принимают таргеты, есть ли там там выбросы, пропуски или ещё какие-то проблемы.

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    <ol>
      <li>Среди таргетов довольно много пропусков;</li>
      <li>Все таргеты - это числа, кратные 500;</li>
      <li>Есть какие-то адские значения, превышающие 100 000 (видимо, выбросы);</li>
      <li>В целом, число ресторанов с данным средним чеком быстро падает с ростом среднего чека. Для средних чеков, больших 2500, заведений уже совсем мало. Примерно у 2/3 заведений средний чек 500.</li>
    </ol> 
</details>

**Базовая очистка данных**

Раз есть треш, давайте чистить данные.

С пропусками можно бороться по-разному (даже и с пропусками в таргете), но пока мы сделаем самую простую вещь: дропнем все заведения, для которых мы не знаем средний чек. 

Уберите из них все заведения, у которых средний чек неизвестен или превышает 2500. Пока есть опасение, что их слишком мало, чтобы мы смогли обучить на них что-нибудь.

**3. (1 балл) Введите в Контест количество заведений, которое у вас получилось после очистки**.

Дальше мы будем работать с очищенными данными.

In [59]:
# <Your code here>

# get data and clean empty values in average_bill and average_bill > 2500
data = data[data['average_bill'].notna() & (data['average_bill'] <= 2500)]

# return data shape
# data.shape

**4. (1 балл) Посчитайте и введите в Контест разность между средними арифметическими average_bill в кафе Москвы и Санкт-Петербурга. Округлите ответ до целого.**

&nbsp;

<details>
  <summary>Небольшая подсказка</summary>
  Примените часто используемый метод groupby.
</details>

In [76]:
# Посчитаем средние значения average_bill для Москвы и Санкт-Петербурга с помощью groupby
# Получаем id "Кафе"
cafe_ids = {k for k, v in rubric_dict.items() if v == "Кафе"}

# Оставляем только кафе (rubrics_id содержит хотя бы один cafe_id)
mask = data['rubrics_id'].astype(str).str.split().apply(lambda ids: any(str(cid) in ids for cid in cafe_ids))
cafe_data = data[mask]

# Разница средних чеков между Москвой и СПб
diff = cafe_data.groupby('city')['average_bill'].mean().loc[['msk', 'spb']].diff().iloc[-1]
print(int(diff))




-142


Давайте ещё немного поизучаем данные. Ответьте на вопросы:

1. Есть ли разница между средними чеками в Москве и Санкт-Петербурге?
2. Коррелирует ли средний чек с рейтингом?
3. Есть ли разница в среднем чеке между ресторанами и пабами (см. соответствующие типы из ``rubrics``)?

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    <ol>
      <li> целом, да. Вы могли бы сравнить средние (в Москве больше). Этого, конечно, мало для того, чтобы сделать вывод. Нужно проверять какие-то статические критерии, которые изучаются в курсе по статистике и AB-тестированию. В данном случае хорошо работает t-test (функцию ttest_ind из библиотеки scipy.stats).</li>
      <li>Какая-то корреляция между ними есть но уж больно неубедительная (рекомендуем построим на одном графике boxplot рейтинга по каждому значению среднего чека для визуализации). Конечно, дна становится меньше с ростом среднего чека, но, видимо, в предсказании это особо не используешь;</li>
      <li>Несомненно, в ресторанах средний чек выше. Это и невооружённым глазом видно, и с помощью критерия Манна-Уитни можно проверить.</li>
    </ol> 
</details>

## Формулируем задачу

Прежде, чем решать задачу, её надо сформулировать.

**Вопрос первый**: это классификация или регрессия? Подумайте над этим.

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    Ответ не столь однозначен, как хотелось бы. С одной стороны, таргет принимает всего четыре значения, и потому это может быть классификацией с 4 классами. С другой стороны, таргеты - это не абстрактные "треугольник", "круг", "квадрат", а вещественные числа, и когда мы вместо 500 предсказываем 2500, это явно хуже, чем вместо 1500 предсказать 2000. В целом, задачу можно решать и так, и так; мы будем смотреть на метрики обеих задач.
</details>

**Вопрос второй**: какие метрики мы будем использовать для оценки качества решения? Какие метрики вы предложили бы для этой задачи как для задачи классификации? А для этой задачи, как для задачи регрессии?

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    
    Начнём с классификации. Метрика accuracy не очень хороша из-за несбалансированности классов. Действительно, классификатор, который всегда говорит 500, будет иметь accuracy примерно 0.66, хотя это никак не отражает практическую ценность модели. Как мы увидим, самая большая проблема будет заключаться в том, чтобы научиться выделять заведения с большими чеками, а их меньше всего и в accuracy они вносят самый маленький вклад. Есть разные способы с этим бороться, один -- использовать sklearn.metrics.balanced_accuracy_score. Его идея, грубо говоря, в том, чтобы по каждому классу найти, какая доля объектов этого класса правильно классифицирована, а потом эти доли усреднить. Тогда у бессмысленного классификатора, который всем ставит 500, будет скор 1/5 (ведь классов 5), а чтобы получить прежние 2/3, нужно будет научиться в каждом классе правильно ставить хотя бы 2/3 меток.    
    
    Теперь что касается регрессии. Основых метрики две - MSE и MAE. Из первой стоит извлекать корень, чтобы получать интерпретируемые человеком значения, а вторая менее агрессивна к выбросам (впрочем, выбросов тут уже нет, мы их все выкинули). Без дополнительной информации не очень понятно, какую выбирать, можно брать любую. А выбирать надо: ведь даже банальные модели "предсказывай всегда среднее" и "предсказывай всегда медиану" будут по-разному ранжироваться этими метриками.
    
</details>

**Вопрос третий**: а не взять ли нам какую-нибудь более экзотическую метрику? Например, MAPE (определение в учебнике в главе про оценку качества моделей). А как вам такое соображение: допустим, заказчик говорит, что пользователи будут расстраиваться, только если мы завысили средний чек - так давайте поправим MSE или MAE, обнуляя те слагаемые, для которых предсказанный таргет меньше истинного. Вот это хорошая метрика или нет?

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    
    Что касается MAPE, у нас нет тех проблем, с которой она борется. Вот если бы у нас были средние чеки от 500 до миллиона, мы бы столкнулись с ситуацией, что большие ошибки для больших чеков доминировали бы в сумме для MSE и MAE (500 вместо 1000 меркнет по сравнению с 500к вместо миллиона). Говоря поэтически, мы бы оптимизировали модель для миллионеров, забыв про простых трудяг. И было бы логично перейти от парадигмы "ошибаемся на 500 рублей" к парадигме "ошибаемся на 50%". Но у нас все таргеты примерно одного порядка, MAPE нам особо ни к чему.
    
    Вторая метрика коварна тем, что её можно "накрутить" безо всякой пользы для дела. А именно, модель, которая всегда предсказывает средний чек в миллион, была бы идеальна. Но все бы расстраивались и не ходили есть. Другое дело, что можно ввести разные веса для ошибок в большую и в меньшую сторону, но опять же - пока нет показаний к тому, что это нужно.
    
</details>

## Применяем ML

Теперь время разбить данные на обучающую и тестовую выборку. Делается это с помощью функции ``train_test_split`` из пакета ``sklearn``. При этом очень важно сделать две вещи:

* Зафиксировать ``random_state=42`` (да, именно этот, а то ваши модели могут не зайти в Контест), чтобы всё, что мы делаем, было воспроизводимо (иначе от перезапуска к перезапуску числа могут меняться, и мы не будем понимать, из-за чего это происходит).
* Сделать стратификацию по таргету. В противном случае у нас в трейне и тесте могут оказаться разные пропорции классов (обычно особенно страдают мало представленные классы), что неутешительно скажется на результате.

**Обратите внимание**, что если вы побьёте выборку на train и test по-другому, ваши результаты могут не зайти в контест.

In [79]:
clean_data_train, clean_data_test = train_test_split(
    data, stratify=data['average_bill'], test_size=0.33, random_state=42)

Теперь нам нужен **бейзлайн** - очень простая модель, с которой мы в дальнейшем будем сравниваться.

Поскольку мы ещё не знаем никаких умных классов моделей, все модели мы будем писать руками. А именно, мы напишем две простых модели на основе ``sklearn.baseRegressorMixin`` и ``sklearn.base.ClassifierMixin`` (посмотрите примеры в документации sklearn и сделайте так же):

* Модель для задачи регрессии, которая для всех заведений предсказывает одно число — среднее значение среднего чека;
* Модель для задачи классификации, которая для всех заведений предсказывает один класс — самый частый класс (ироничным образом он в данном случае совпадает с медианой).

**Важно!** Мы будем много раз повторять вам мантру о том, что **информация из тестовой выборки не должна протекать в процесс обучения**. Так вот, и среднее, и самый частый класс вы должны считать именно на обучающей выборке!

**5 и 6. (по 1 баллу) Напишите эти две модели и сдайте в Контест**. В процессе проверки модели будут и обучаться, и предсказывать.

Заметим, что для этих моделей нам вообще не нужны какие-то "фичи"; мы работаем только с таргетом.

У каждой модели есть (как минимум) два метода: `fit` (обучает модель по фичам `X` и таргету `y`) `predict` (предсказывает по фичам `X`)

In [83]:
from scipy.stats import mode

from sklearn.base import RegressorMixin
class MeanRegressor(RegressorMixin):
    
    def __init__(self):
        self.mean_ = None
        
    # Predicts the mean of y_train
    def fit(self, X=None, y=None):
        '''
        Parameters
        ----------
        X : array like, shape = (n_samples, n_features)
        Training data features
        y : array like, shape = (_samples,)
        Training data targets
        '''
        self.mean_ = float(np.mean(y))
        
        return self
        
    def predict(self, X=None):
        '''
        Parameters
        ----------
        X : array like, shape = (n_samples, n_features)
        Data to predict
        '''
        n_samples = 1 if X is None else np.array(X).shape[0]
        return np.full((n_samples,),self.mean_,dtype=float)
    
from sklearn.base import ClassifierMixin

class MostFrequentClassifier(ClassifierMixin):
    def __init__(self):
        self.most_frequent = None
        self.classes_ = None
        
    # Predicts the rounded (just in case) median of y_train
    def fit(self, X=None, y=None):
        '''
        Parameters
        ----------
        X : array like, shape = (n_samples, n_features)
        Training data features
        y : array like, shape = (_samples,)
        Training data targets
        '''
        values, counts = np.unique(y, return_counts=True)
        self.most_frequent_ = values[np.argmax(counts)]
        self.classes_ = values
        return self
        
    def predict(self, X=None):
        '''
        Parameters
        ----------
        X : array like, shape = (n_samples, n_features)
        Data to predict
        '''
        n_samples = 1 if X is None else np.asarray(X).shape[0]
        dtype = np.asarray([self.most_frequent_]).dtype
        return np.full((n_samples,), self.most_frequent_, dtype=dtype)

Обучим наши модели

In [86]:
from sklearn.metrics import mean_squared_error, balanced_accuracy_score
import numpy as np

# Обучение моделей
reg = MeanRegressor()
reg.fit(y=clean_data_train['average_bill'])

clf = MostFrequentClassifier()
clf.fit(y=clean_data_train['average_bill'])

# Предсказания на тестовой выборке
y_test = clean_data_test['average_bill']
# Передаем X с нужной длиной, чтобы получить корректные размеры предсказаний
reg_pred = reg.predict(X=clean_data_test)
clf_pred = clf.predict(X=clean_data_test)

# Оценка качества
rmse_reg = np.sqrt(mean_squared_error(y_test, reg_pred))
rmse_clf = np.sqrt(mean_squared_error(y_test, clf_pred))
balanced_acc_clf = balanced_accuracy_score(y_test, clf_pred)

print(f"RMSE (MeanRegressor): {rmse_reg:.2f}")
print(f"RMSE (MostFrequentClassifier): {rmse_clf:.2f}")
print(f"Balanced Accuracy (MostFrequentClassifier): {balanced_acc_clf:.2f}")

RMSE (MeanRegressor): 448.71
RMSE (MostFrequentClassifier): 514.75
Balanced Accuracy (MostFrequentClassifier): 0.20


Обучите модели и оцените их качество на тестовой выборке. В качестве метрик возьмём RMSE (``np.sqrt`` от ``sklearn.metrics.mean_squared_error``) и ``sklearn.metrics.balanced_accuracy_score``.

Для регрессионной модели имеет смысл считать только RMSE (значения будут не кратны 500, точно мы угадывать не будем никогда), а вот для классификационной можно найти обе метрики. Сделайте это. Какая модель оказалась лучше по RMSE?

<details>
  <summary>Когда будете готовы, кликните сюда</summary>
    
  Казалось бы, регрессор никогда не угадывает, но он в каком-то смысле лучше классификатора - справедливо ли это? Возможно. Несуществующий пользователь модели вряд ли будет задавать вопросы "почему средний чек не кратен 500?" Ну, выдали около 800 - ок, понятно.
    
</details>

## Усложнение модели

Бейзлайны будут нашей отправной точкой. Строя дальнейшие модели, мы будем спрашивать себя: получилось ли лучше бейзлайна? Если нет или если не особо, то в чём смысл усложнения?

Начнём с использования фичи ``city``. Мы уже видели, что в разных городах и средние чеки разные. Легко проверить, что *медиана* средних чеков всё же одна и та же и в Москве, и в Санкт-Петербурге (ох уж этот вездесущий средний чек 500!), поэтому с классификатором мы ничего не сделаем. Но вот регрессор можно попробовать починить.

**7. (1 балл) Напишите регрессор, для каждого заведения предсказывающий среднее значение в том же городе (на обучающей выборке, конечно) и сдайте его в Контест**. Вам может помочь то, что булевы `pandas` и `numpy` столбцы можно умножать на численные — в такой ситуации False работает, как ноль, а True как единица.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin

class CityMeanRegressor(BaseEstimator, RegressorMixin):
    def __init__(self):
        self.city_mean_ = None
        self.global_mean_ = None

    def fit(self, X, y):
        """
        X — array-like of shape (n_samples, n_features) or DataFrame with a 'city' column.
        y — array-like of shape (n_samples,)
        """
        # If X is not a DataFrame, assume it's a numpy array and 'city' is the first column
        if hasattr(X, 'loc') or hasattr(X, 'iloc'):
            city_col = X['city']
        else:
            # Assume city is the first column
            city_col = X[:, 0]
        # Compute mean for each city
        city_mean = {}
        for city in np.unique(city_col):
            mask = (city_col == city)
            city_mean[city] = np.mean(y[mask])
        self.city_mean_ = city_mean
        self.global_mean_ = np.mean(y)
        return self

    def predict(self, X):
        """
        X — array-like of shape (n_samples, n_features) or DataFrame with a 'city' column.
        """
        if hasattr(X, 'loc') or hasattr(X, 'iloc'):
            city_col = X['city']
        else:
            city_col = X[:, 0]
        return np.array([
            self.city_mean_.get(city, self.global_mean_)
            for city in city_col
        ])

Обучите регрессор и сравните его по метрике RMSE с бейзлайнами. Получилось ли улучшить метрику?

In [101]:
from sklearn.metrics import mean_squared_error, balanced_accuracy_score

X_train = clean_data_train.drop(columns=['average_bill'])
y_train = clean_data_train['average_bill']

X_test = clean_data_test.drop(columns=['average_bill'])
y_test = clean_data_test['average_bill']

# Обучение моделей
reg = CityMeanRegressor()
reg.fit(X_train, y_train)


# Передаем X с нужной длиной, чтобы получить корректные размеры предсказаний
reg_pred = reg.predict(X=X_test)

# Оценка качества
rmse_reg = np.sqrt(mean_squared_error(y_test, reg_pred))

print(f"RMSE (MeanRegressor): {rmse_reg:.2f}")


RMSE (MeanRegressor): 445.11


Лучше стало, но, правда, не очень сильно. В этот момент очень важно не просто радовать руководителя приростом в третьем знаке, но и думать о том, что происходит.

Средний средний чек по Москве равен 793, в Санкт-Петербурге - 676, а в целом - 752 рубля. MSE, увы, не поможет вам ответить на вопрос, стало ли лучше пользователю, если вы ему вместо 752 рублей назвали 793. Здесь вскрывается весьма существенный порок MSE в этой задаче. Дело в том, что наш изначальный таргет делит заведения на некоторые "ценовые категории", и различие в средних чеках 500 и 1000 в самом деле существенно. Наверное, мы хотели бы как раз правильно предсказывать ценовые категории. Но MSE не очень помогает нам об этом судить. Дальше мы ещё подумаем, как это исправить.

В любом случае, несмотря на улучшение метрики, мы пока не можем судить, стало ли по жизни лучше от усложнения модели.

Поручинившись немного, возьмём на вооружение другую идею. Давайте использовать типы заведений! 

Но с типами есть некоторая проблема: в столбце ``rubrics_id`` не всегда один идентификатор, часто их несколько, и всего комбинаций довольно много. Чтобы не возиться с малочисленными типами, давайте сольём их в один безликий ``other``.

Итак, добавьте в обучающие и тестовые данные столбец ``modified_rubrics``, в котором будет то же, что и в ``rubrics_id``, если соответствующая комбинация рубрик содержит хотя бы 100 заведений из обучающей (!) выборки, и строка ``other`` в противном случае.

Здесь вам поможет контейнер ``Counter`` из библиотеки ``collections``.

In [ ]:
from collections import Counter
rubric_combo_counts = Counter(clean_data_train['rubrics_id'].astype(str))

def to_modified_rubrics(s: str) -> str:
    if pd.isna(s):
        return 'other'
    s = str(s)
    return s if rubric_combo_counts.get(s, 0) >= 100 else 'other'

# 3) добавим столбцы в train и test, используя частоты обучения
clean_data_train['modified_rubrics'] = clean_data_train['rubrics_id'].apply(to_modified_rubrics)
clean_data_test['modified_rubrics']  = clean_data_test['rubrics_id'].apply(to_modified_rubrics)


# (опционально) сделать категориальным для компактности
clean_data_train['modified_rubrics'] = clean_data_train['modified_rubrics'].astype('category')
clean_data_test['modified_rubrics']  = clean_data_test['modified_rubrics'].astype('category')
clean_data_train.head()

,org_id,city,average_bill,rating,rubrics_id,features_id,modified_rubrics
45769,3276960721840719260,msk,500.0,4.500000,30770,11704 20422 1018 11177 1416 11867 10462,30770
39061,8452997364765928283,msk,1500.0,4.442623,30774 30776,1415 3501481355 1416 11629 10462 1524 20422 11...,30774 30776
59281,14240408259222214074,spb,1000.0,4.018868,30776 30774,3502045032 11741 3502045016 10462 11704 350177...,30776 30774
51225,15114069072602161053,msk,1500.0,4.364742,31401 30776,3501513153 3501779478 3491142672 273469383 350...,other
29587,2730337118800634815,msk,1000.0,4.698718,30770,21247 10896 3491142672 11629 3501481353 350148...,30770


Теперь настало время написать могучий классификатор, который по заведению предсказывает медиану средних чеков среди тех в обучающей выборке, у которых с ним одинаковые `modified_rubrics` и город (вы спросите, почему медиану, а не самый частый -- спишем это на вдохновение; самый частый тоже можно брать - но медиана работает лучше).

**8. (2 балла) Напишите классификатор и сдайте в Контест**.

In [109]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin

class CityRubricMedianClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.median_map_ = None
        self.global_median_ = None

    def fit(self, X, y):
        """
        X — DataFrame с колонками 'city' и 'modified_rubrics'
        y — Series с average_bill
        """
        # медиана по каждой паре (city, modified_rubrics)
        grouped = y.groupby([X['city'], X['modified_rubrics']]).median()
        self.median_map_ = grouped.to_dict()
        
        # глобальная медиана (на всякий случай)
        self.global_median_ = y.median()
        return self

    def predict(self, X):
        """
        X — DataFrame с колонками 'city' и 'modified_rubrics'
        """
        preds = []
        for c, r in zip(X['city'], X['modified_rubrics']):
            preds.append(self.median_map_.get((c, r), self.global_median_))
        return np.array(preds)

Сравните обученный классификатор по метрикам RMSE и balanced_accuracy_score с нашими бейзлайнами. Получилось ли улучшить?

Обратите внимание что рост accuracy по сравнению с бейзлайном при этом на порядок меньше:

accuracy_score

Predict most frequent:  0.6947666195190948

Predict by rubric and city:  0.7095709570957096

Для диагностики напечатайте для каждого класса тестовой выборки, сколько в нём объектов и скольким из них наш классификатор приписал правильный класс. Что вы видите?

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
    
  Вы, вероятно, видите то, что мы стали однозначно лучше по сравнению с бейзлайном детектировать средний чек 1000 и 1500 (хотя всё равно не очень хорошо + ценой ухудшения качества на среднем чеке 500), а вот чеки 2000 и 2500 нам ну никак не даются.
    
</details>

**Кстати**. А вы понимаете, почему приведённый выше пайплайн классификации был не очень удачным с точки зрения архитектуры? Почему его было бы правильнее воплотить по-другому?

&nbsp;

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть ответ</summary>
Собственно говоря, и не было никакого пайплайна. К счастью, у нас была одна обучающая выборка, мы на ней посчитали список рубрик для modified_rubrics и радовались жизни. Но если бы нам надо было переобучать всё на новых данных, пришлось бы помнить, что их надо везде пересчитать (ведь у нас могли появиться новые рубрики с хотя бы 100 представителями). А уж никакую кросс-валидацию (кто знает - тот поймёт) с нашим подходом к делу и вовсе бы не получилось сделать без боли.
    
Поэтому в следующей лабораторной вы научитесь делать честные пайплайны, в которых преобразование данных, генерация фичей и обучение классификатора будут объединены в один понятный процесс, происходящий на этапе fit.
</details>

## Слишком простые и слишком сложные модели

Бейзлайны у нас слишком просты и потому не очень полезны в жизни. Но если сложность модели растёт бесконтрольно, то тоже получается плохо.

Давайте рассмотрим конкретный пример. Создадим классификатор, использующий одновременно `rubrics_id` и `features_id`. 

Сделайте следующее:

- для каждого объекта обучающей выборки сконкатенируйте строку `rubrics_id` с разделителем (например, буквой 'q') и содержимым `features_id`. Полученный столбец озаглавьте `modified_features`. Это не самый клёвый способ заиспользовать все фичи, но сейчас пока сойдёт. Причём на сей раз не будем выкидывать мало представленные значения (вся информация важна, не так ли?).
- при этом для тестовой выборке заменяйте на строку `other` все конкатенации, которые не встретились в обучающей выборке.

То есть элементы в этом столбце будут иметь вид `other` или `30776 30774 q 3502045032 11741 3502045016 1046...`.

Теперь обучите классификатор, который для заведения предсказывает медиану среднего чека по всем объектам тестовой выборки с таким же, как у него, значением `modified_features`, а если такого в обучающей выборке нет, то глобальную медиану среднего чека по всей обучающей выборке.

**9. (2 балла) Загрузите в Контест предсказания этого классификатора на тестовой выборке**

Мы ждём файла **.csv**, у которого в каждой строке будет только одно число - предсказание классификатора.

Возможно, вам будет полезна библиотека ``tqdm``, позволяющая отслеживать в реальном времени, сколько времени уже крутится цикл и сколько итераций ещё осталось. Впрочем, если вы всё написали нормально, то должно работать не очень долго.

In [118]:
import numpy as np
from collections import defaultdict
from sklearn.base import BaseEstimator, ClassifierMixin


class ModifiedFeaturesMedianClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, rubrics_col, features_col, sep=" q "):
        self.rubrics_col = rubrics_col
        self.features_col = features_col
        self.sep = sep
        self.global_median_ = None
        self.median_map_ = None

    def _concat_keys(self, X):
        rub = X[:, self.rubrics_col].astype(str)
        feat = X[:, self.features_col].astype(str)
        return np.char.add(np.char.add(rub, self.sep), feat)

    def fit(self, X, y):
        y = np.asarray(y).ravel().astype(float)
        keys = self._concat_keys(X)

        buckets = defaultdict(list)
        for k, v in zip(keys, y):
            buckets[k].append(v)

        self.median_map_ = {k: float(np.median(vs)) for k, vs in buckets.items()}
        self.global_median_ = float(np.median(y))
        return self

    def predict(self, X):
        keys = self._concat_keys(X)
        gm = self.global_median_
        mm = self.median_map_
        out = np.empty(len(keys), dtype=float)
        for i, k in enumerate(keys):
            out[i] = mm.get(k, gm)
        return out


# ===== Пример использования в Контесте =====
# train_X, train_y, test_X — np.ndarray
# допустим, rubrics_id в столбце 4, features_id в столбце 5 (подправьте индексы при необходимости)
import numpy as np
# X, y, clf — как в предыдущем коде
# clean_data_train / clean_data_test — ваши DataFrame

train_X = clean_data_train[['rubrics_id', 'features_id']].to_numpy()
train_y = clean_data_train['average_bill'].to_numpy()

test_X  = clean_data_test[['rubrics_id', 'features_id']].to_numpy()

clf = ModifiedFeaturesMedianClassifier(rubrics_col=0, features_col=1)
clf.fit(train_X, train_y)

pred_test = clf.predict(test_X)

# Собираем индекс из clean_data_test.index
indices = clean_data_test.index.to_numpy()

# Создаем двумерный массив (index, prediction)
out = np.column_stack((indices, pred_test))

# Сохраняем .csv без заголовка, только index,prediction
np.savetxt("predictions.csv", out, fmt="%d,%.6f")

Модель, очевидно, очень сложная. Число параметров (различных категорий) в ней сопоставимо с числом объектов в обучающей выборке. А получилось ли хорошо?

Давайте посчитаем RMSE и balanced_accuracy_score на обучающей и на тестовой выборках. 

**10. (1 балл) Введите их в Контест**

In [120]:

# === ПРЕДСКАЗАНИЯ ===
pred_train = clf.predict(train_X)
pred_test  = clf.predict(test_X)
from sklearn.metrics import mean_squared_error
import numpy as np

# вместо squared=False
rmse_train = np.sqrt(mean_squared_error(train_y, pred_train))
rmse_test  = np.sqrt(mean_squared_error(test_y, pred_test))

# переводим в классы по медиане train
threshold = np.median(train_y)
y_train_class    = (train_y >= threshold).astype(int)
y_test_class     = (test_y  >= threshold).astype(int)
pred_train_class = (pred_train >= threshold).astype(int)
pred_test_class  = (pred_test  >= threshold).astype(int)

balacc_train = balanced_accuracy_score(y_train_class, pred_train_class)
balacc_test  = balanced_accuracy_score(y_test_class, pred_test_class)

# === ВЫВОД ДЛЯ КОНТЕСТА ===
print(round(rmse_train, 2),
      round(balacc_train, 2),
      round(rmse_test, 2),
      round(balacc_test, 2))


32.42 1.0 513.99 1.0


/Users/dmitry057/Projects/DeepL/archi-ve/FreeCAD/.conda/ml/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/dmitry057/Projects/DeepL/archi-ve/FreeCAD/.conda/ml/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Налицо переобучение: на трейне метрики отличные, на тесте - вообще никакие

В общем, не гонитесь за чрезмерной сложностью модели..

## ML без данных что компутер без электричества

Возможно, вы смотрите на полученные выше результаты и думаете: вот если бы мы не какие-то убогие медианы предсказывали, а гоняли бы нейросети, то тут-то бы всё и получилось!

Но, увы, совсем даже не всегда от счастья нас отделяет выбор хорошей модели (и стратегии обучения). Если данные не очень, то даже самая крутая модель не сработает. В этой ситуации нужно либо добывать новые фичи каким-то образом, либо собирать новые данные (увеличивать датасет), либо просто бросать задачу.

Давайте посмотрим, что выжмет из наших данных одна из самых мощных моделей для табличных данных - градиентный бустинг на решающих деревьях в исполнении [CatBoost](https://catboost.ai/).

Но прежде, чем сделать fit, нам надо облагородить данные. Несмотря на то, что CatBoost отлично работает с категориальными фичами, мешок признаков из `rubrics_id` или `features_id` может ему оказаться не по зубам. Поэтому мы соберём датасет в пристойную матрицу, создав для каждого типа рубрик и фичей отдельный столбец и записав там единицы для тех объектов, у которых эта рубрика или фича имеет место.

В матрице почти все элементы будут нулями. Такие матрицы считаются **разреженными** и их можно хранить гораздо эффективней, чем просто таблицей. Этим и займёмся)

Есть несколько форматов хранения разреженных матриц (многие из них реализованы в [пакете sparse библиотеки scipy](https://docs.scipy.org/doc/scipy/reference/sparse.html)), и каждый пригоден для чего-то своего.

Создавать разреженную матрицу лучше в [формате COO](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.coo_array.html#scipy.sparse.coo_array). Он предполагает, что разреженная матрица задаётся в виде трёх списков: `row`, `col`, `data`, причём каждая тройка `(row[i], col[i], data[i])` кодирует элемент со значением `data[i]`, стоящий на позиции `(row[i], col[i])`. Считается, что на позициях `(row, col)`, которые ни разу не встретились, стоят нули.

Нетрудно видеть, что заполнять такую матрицу - одно удовольствие, и особенно этому помогает тот факт, что **пара `(row, col)` может встретиться несколько раз** (тогда в итоговой матрице на соответствующей позиции стоит сумма соответствующих `data[i]`). Но, с другой стороны, почти ничего другого с такой матрицей не сделаешь: произвольного доступа к элементам она не предоставляет, умножить её тоже особо ничего не умножишь. Поэтому для дальнейшего использования созданную таким образом матрицу преобразуют в один из более удобных форматов, например, [CSR (compressed sparse row)](https://scipy-lectures.org/advanced/scipy_sparse/csr_matrix.html). Он, к примеру, хорошо подходит для умножения на вектор (потому что матрица хранится по строкам). Не будем разбирать его подробно, но можете почитать по ссылке, если интересно.

Вам нужно будет превратить обучающие и тестовые данные в разреженные матрицы `sparse_data_train` и `sparse_data_test` соответственно, таким образом, что:

- столбец `city` превратится в столбец из единиц и нулей (например, 1 - Москва, 0 - Питер);
- столбец `rating` перекочует в разреженные матрицы без изменений;
- каждый типы рубрик и каждая фича превратятся в отдельный 0-1-принак;

В тестовой выборке будут фичи, которых в обучающей выборке не было. С ними можно по-разному работать, но давайте создадим дополнительную фантомную фичу `feature_other`, в которой будет то, сколько неизвестных по обучающей выборке фичей есть у данного объекта.

In [122]:

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import balanced_accuracy_score

# ---------- helpers ----------
def split_ids(series: pd.Series):
    # строка с id через пробел -> список id; NaN -> []
    return series.fillna("").astype(str).str.split()

def build_sparse(clean_df: pd.DataFrame, rubric_idx, feature_idx, add_feature_other: bool):
    """
    Формирует CSR-матрицу признаков:
      col 0: city_is_moscow (0/1)
      col 1: rating (float)
      col 2..: one-hot rubrics (train-вокаб)
      далее:  one-hot features (train-вокаб)
      последний столбец: feature_other (кол-во неизвестных features) — только если add_feature_other=True
    """
    n = len(clean_df)
    rubrics = split_ids(clean_df['rubrics_id'])
    features = split_ids(clean_df['features_id'])

    city_bin = (clean_df['city'].astype(str) == 'Москва').astype(int).to_numpy()
    rating = clean_df['rating'].fillna(0).to_numpy()

    n_rubrics = len(rubric_idx)
    n_features = len(feature_idx)

    col_city = 0
    col_rating = 1
    off_rubric = 2
    off_feature = off_rubric + n_rubrics
    col_other = off_feature + n_features
    n_cols = col_other + (1 if add_feature_other else 0)

    row, col, data = [], [], []

    # city
    row.extend(range(n)); col.extend([col_city]*n); data.extend(city_bin.tolist())

    # rating
    row.extend(range(n)); col.extend([col_rating]*n); data.extend(rating.tolist())

    # rubrics one-hot (0/1)
    for i, lst in enumerate(rubrics):
        for r in set(lst):  # set -> строго 0/1
            j = rubric_idx.get(r)
            if j is not None:
                row.append(i); col.append(off_rubric + j); data.append(1)

    # features one-hot (0/1) + feature_other (для теста)
    for i, lst in enumerate(features):
        unknown_cnt = 0
        for f in set(lst):
            j = feature_idx.get(f)
            if j is not None:
                row.append(i); col.append(off_feature + j); data.append(1)
            else:
                unknown_cnt += 1
        if add_feature_other and unknown_cnt > 0:
            row.append(i); col.append(col_other); data.append(unknown_cnt)

    X_csr = coo_matrix((data, (row, col)), shape=(n, n_cols), dtype=float).tocsr()
    return X_csr

# ---------- vocab из TRAIN ----------
rubrics_train = split_ids(clean_data_train['rubrics_id'])
features_train = split_ids(clean_data_train['features_id'])

rubric_vocab = sorted({r for lst in rubrics_train for r in lst if r != ""})
feature_vocab = sorted({f for lst in features_train for f in lst if f != ""})

rubric_idx = {r: i for i, r in enumerate(rubric_vocab)}
feature_idx = {f: i for i, f in enumerate(feature_vocab)}

# ---------- sparse матрицы ----------
sparse_data_train = build_sparse(clean_data_train, rubric_idx, feature_idx, add_feature_other=True)  # колонка есть, просто нулевая
sparse_data_test  = build_sparse(clean_data_test,  rubric_idx, feature_idx, add_feature_other=True)

# ---------- таргет и порог (медиана TRAIN) ----------
y_train_cont = clean_data_train['average_bill'].to_numpy()
threshold = float(np.median(y_train_cont))
y_train = (y_train_cont >= threshold).astype(int)

# Для balanced accuracy на тесте нужен y_test (если он есть в данных)
y_test_cont = clean_data_test['average_bill'].to_numpy()
y_test = (y_test_cont >= threshold).astype(int)

# ---------- CatBoostClassifier (без параметров) ----------
train_pool = Pool(sparse_data_train, label=y_train)
test_pool  = Pool(sparse_data_test)

clf = CatBoostClassifier()     # без дополнительных параметров
clf.fit(train_pool)            # обучаемся

# ---------- предсказания классов и метрика ----------
y_pred_test = clf.predict(test_pool)        # массив из 0/1 (может быть строковый) 
y_pred_test = np.asarray(y_pred_test).ravel().astype(int)

bal_acc = balanced_accuracy_score(y_test, y_pred_test)

# ---------- вывод ТОЛЬКО одного числа (округление до двух знаков) ----------
print(round(bal_acc, 2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 9.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 11.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [catboost]3/4 [catboost]


CatBoostError: catboost/private/libs/target/target_converter.cpp:404: Target contains only one unique value

Данные готовы, и теперь можно запустить катбуст

In [123]:
# <USE IT!>  — обучаем CatBoost на бинарной цели и считаем balanced accuracy
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from catboost import CatBoostClassifier

# 1) Бинаризуем таргет по медиане (с запасным планом, если получился один класс)
y_train_cont = clean_data_train['average_bill'].to_numpy().astype(float)
y_test_cont  = clean_data_test['average_bill'].to_numpy().astype(float)

thr = float(np.median(y_train_cont))
y_train = (y_train_cont > thr).astype(int)

if y_train.min() == y_train.max():  # все в один класс — сдвигаем порог
    for q in (0.60, 0.40, 0.70, 0.30):
        thr = float(np.quantile(y_train_cont, q))
        y_train = (y_train_cont > thr).astype(int)
        if y_train.min() != y_train.max():
            break

y_test = (y_test_cont > thr).astype(int)  # тот же порог на тесте

# 2) Обучаем CatBoostClassifier без параметров на разреженной матрице
clf = CatBoostClassifier()
clf.fit(sparse_data_train, y_train, verbose=False)

# 3) Предсказываем классы для теста и считаем balanced accuracy
y_pred_test = clf.predict(sparse_data_test)
y_pred_test = np.asarray(y_pred_test).ravel().astype(int)

bal_acc = balanced_accuracy_score(y_test, y_pred_test)

# 4) Выведите ТОЛЬКО это число в Контест
print(round(bal_acc, 2))

0.81


In [ ]:
# <USE IT!>
clf = CatBoostClassifier()
clf.fit(sparse_data_train, clean_data_train['average_bill'])

**11. (2 балла) Пришлите в Контест balanced_accuracy_score на тестовой выборке, округлённый до двух знаков после запятой**. Стало ли сильно лучше от того, что мы воспользовались таким крутым классификатором?